# MTH9877 — Interest Rate and Credit Models
## Assignment 3: Mortgage Prepayment and Credit Risk Modeling
**Baruch College, CUNY | Prof. Andrew Lesniewski | Spring 2026**

Data: Freddie Mac Single-Family Loan-Level Dataset (1999–2025)  
Focus: 30-year fixed-rate mortgages, prepayment and default competing risks

## Step 0 — Environment Setup

In [ ]:
import subprocess, sys
pkgs = ["polars", "lifelines", "lightgbm", "fredapi"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("All packages installed.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE = Path("/Users/yueqilin/Desktop/MTH9877 IR/IR&C Assignment3")
ORIG_DIR  = BASE / "Origination_Historical_Data"
PERF_DIR  = BASE / "Monthly_Performance_historical_data_time"
OUT_DIR   = BASE / "processed"
OUT_DIR.mkdir(exist_ok=True)

SURVIVAL_PATH = OUT_DIR / "survival_loans.parquet"
PANEL_PATH    = OUT_DIR / "panel_monthly.parquet"
MACRO_PATH    = OUT_DIR / "macro_monthly.parquet"

FRED_API_KEY = "5c9f565b51951d5e154c4480a0cedd37"

print(f"BASE : {BASE}")
print(f"ORIG : {ORIG_DIR}")
print(f"PERF : {PERF_DIR}")
print(f"OUT  : {OUT_DIR}")

## Step 1 — Build Survival Dataset (`survival_loans.parquet`)

For each 30-year FRM loan we need one row:

| column | description |
|--------|-------------|
| `duration` | months from origination to prepayment / default / last observation |
| `prepaid` | 1 if ZeroBalanceCode == 1 (full payoff), else 0 |
| `defaulted` | 1 if ZeroBalanceCode in {2,3,9}, else 0 |
| loan static features | FICO, LTV, rate, DTI, purpose, occupancy, state, vintage |

**Strategy:** scan origination files to build the loan universe, then scan performance files year-by-year (lazy) to extract event timing. Join on `LoanSequenceNumber`.

In [ ]:
def build_survival_dataset(force_rebuild: bool = False) -> pl.DataFrame:
    """
    Scans all origination + performance parquet files and produces one row
    per 30-yr FRM loan with survival outcome columns.
    Results are cached to SURVIVAL_PATH.
    """
    if SURVIVAL_PATH.exists() and not force_rebuild:
        print(f"Loading cached survival dataset from {SURVIVAL_PATH}")
        return pl.read_parquet(SURVIVAL_PATH)

    # ── 1. Origination universe ────────────────────────────────────────────────
    print("Step 1/3  Reading origination files …")
    orig_files = sorted(ORIG_DIR.glob("historical_data_[12]*.parquet"))
    print(f"  Found {len(orig_files)} origination files")

    orig_cols = [
        "LoanSequenceNumber", "CreditScore", "OriginalLoantoValueLTV",
        "OriginalInterestRate", "OriginalDebttoIncomeRatio", "OriginalUPB",
        "LoanPurpose", "OccupancyStatus", "PropertyState", "PropertyType",
        "AmortizationType", "OriginalLoanTerm", "FirstPaymentDate",
        "NumberofBorrowers", "Channel",
    ]
    orig = (
        pl.scan_parquet(orig_files)
        .select(orig_cols)
        .filter(
            (pl.col("AmortizationType") == "FRM") &
            (pl.col("OriginalLoanTerm") == 360)
        )
        # derive vintage year from FirstPaymentDate (YYYYMM)
        .with_columns(
            (pl.col("FirstPaymentDate") // 100).alias("VintageYear")
        )
        # clean sentinel values
        .with_columns([
            pl.when(pl.col("CreditScore") > 850).then(None).otherwise(pl.col("CreditScore")).alias("CreditScore"),
            pl.when(pl.col("OriginalDebttoIncomeRatio") >= 999).then(None).otherwise(pl.col("OriginalDebttoIncomeRatio")).alias("OriginalDebttoIncomeRatio"),
            pl.when(pl.col("OriginalLoantoValueLTV") >= 999).then(None).otherwise(pl.col("OriginalLoantoValueLTV")).alias("OriginalLoantoValueLTV"),
        ])
        .collect()
    )
    print(f"  FRM-360 loans: {orig.height:,}")

    # ── 2. Performance: extract event per loan ─────────────────────────────────
    # ZeroBalanceCode: 1=prepaid, 2/3/9=default, null=active
    PREPAY_CODE  = 1
    DEFAULT_CODES = {2, 3, 9}

    print("Step 2/3  Scanning performance files year-by-year …")
    perf_files = sorted(PERF_DIR.glob("historical_data_time_[12]*.parquet"))
    print(f"  Found {len(perf_files)} performance files")

    perf_cols = ["LoanSequenceNumber", "LoanAge", "ZeroBalanceCode"]
    perf_parts = []

    for f in perf_files:
        chunk = (
            pl.scan_parquet(f)
            .select(perf_cols)
            # keep only rows that matter: events OR last-observed month
            # we'll aggregate to get (max_age, first_event_age, first_event_code)
            .group_by("LoanSequenceNumber")
            .agg([
                pl.col("LoanAge").max().alias("MaxLoanAge"),
                pl.col("LoanAge").filter(pl.col("ZeroBalanceCode").is_not_null()).min().alias("EventLoanAge"),
                pl.col("ZeroBalanceCode").filter(pl.col("ZeroBalanceCode").is_not_null()).sort_by(
                    pl.col("LoanAge").filter(pl.col("ZeroBalanceCode").is_not_null())
                ).first().alias("EventCode"),
            ])
            .collect()
        )
        perf_parts.append(chunk)
        print(f"  {f.name}: {chunk.height:,} loans", end="\r")

    print()
    perf_all = pl.concat(perf_parts)

    # A loan may appear across multiple year files — take the one with the event,
    # or the longest observation window if still active.
    perf_summary = (
        perf_all
        .group_by("LoanSequenceNumber")
        .agg([
            pl.col("MaxLoanAge").max(),
            pl.col("EventLoanAge").drop_nulls().min().alias("EventLoanAge"),
            pl.col("EventCode").drop_nulls().first().alias("EventCode"),
        ])
        .with_columns([
            # duration = age at event if event observed, else max observed age
            pl.when(pl.col("EventLoanAge").is_not_null())
              .then(pl.col("EventLoanAge"))
              .otherwise(pl.col("MaxLoanAge"))
              .alias("duration"),
            # event indicators
            (pl.col("EventCode") == PREPAY_CODE).cast(pl.Int8).alias("prepaid"),
            pl.col("EventCode").is_in(list(DEFAULT_CODES)).cast(pl.Int8).alias("defaulted"),
        ])
        .select(["LoanSequenceNumber", "duration", "prepaid", "defaulted"])
    )
    print(f"  Performance summary: {perf_summary.height:,} unique loans")

    # ── 3. Join ────────────────────────────────────────────────────────────────
    print("Step 3/3  Joining origination + performance …")
    survival = (
        orig
        .join(perf_summary, on="LoanSequenceNumber", how="left")
        # loans with no performance record: treat as censored at duration=0 (drop)
        .filter(pl.col("duration").is_not_null() & (pl.col("duration") > 0))
        # fill missing event flags
        .with_columns([
            pl.col("prepaid").fill_null(0),
            pl.col("defaulted").fill_null(0),
        ])
        .drop(["AmortizationType", "OriginalLoanTerm"])
    )

    survival.write_parquet(SURVIVAL_PATH)
    print(f"  Saved {survival.height:,} loans → {SURVIVAL_PATH}")
    return survival

In [ ]:
# Run the pipeline (takes ~20-40 min on first run; cached after that)
survival = build_survival_dataset(force_rebuild=False)
print(survival.schema)
print(survival.describe())

In [ ]:
# Quick sanity checks
total   = survival.height
prepaid = survival["prepaid"].sum()
default = survival["defaulted"].sum()
censored = total - prepaid - default
print(f"Total loans  : {total:>12,}")
print(f"Prepaid      : {prepaid:>12,}  ({prepaid/total:.1%})")
print(f"Defaulted    : {default:>12,}  ({default/total:.1%})")
print(f"Censored     : {censored:>12,}  ({censored/total:.1%})")
print(f"\nMedian duration (months): {survival['duration'].median():.0f}")
print(f"Duration range          : {survival['duration'].min()} – {survival['duration'].max()}")
print(f"\nVintage range: {survival['VintageYear'].min()} – {survival['VintageYear'].max()}")

## Step 2 — Fetch FRED Macro Covariates

Monthly series 1999–2025:
- `MORTGAGE30US` — 30-year fixed mortgage rate  
- `UNRATE` — civilian unemployment rate  
- `CPIAUCSL` — CPI all urban consumers (YoY % change = inflation)  
- `CSUSHPISA` — Case-Shiller national home price index (YoY % change)

In [ ]:
def fetch_macro(force_rebuild: bool = False) -> pl.DataFrame:
    if MACRO_PATH.exists() and not force_rebuild:
        print(f"Loading cached macro data from {MACRO_PATH}")
        return pl.read_parquet(MACRO_PATH)

    from fredapi import Fred
    fred = Fred(api_key=FRED_API_KEY)

    series = {
        "mortgage_rate": "MORTGAGE30US",
        "unemployment":  "UNRATE",
        "cpi":           "CPIAUCSL",
        "hpi":           "CSUSHPISA",
    }

    frames = {}
    for name, sid in series.items():
        s = fred.get_series(sid, observation_start="1999-01-01", observation_end="2025-12-31")
        s = s.resample("MS").last().ffill()        # forward-fill weekly → monthly
        frames[name] = s
        print(f"  {sid}: {len(s)} obs  ({s.index[0].date()} – {s.index[-1].date()})")

    macro_pd = pd.DataFrame(frames)
    macro_pd.index.name = "date"
    macro_pd = macro_pd.reset_index()
    macro_pd["yyyymm"] = (macro_pd["date"].dt.year * 100 + macro_pd["date"].dt.month).astype(int)

    # YoY % changes for CPI and HPI (more economically meaningful)
    macro_pd["cpi_yoy"]  = macro_pd["cpi"].pct_change(12) * 100
    macro_pd["hpi_yoy"]  = macro_pd["hpi"].pct_change(12) * 100

    macro = pl.from_pandas(macro_pd[["yyyymm", "mortgage_rate", "unemployment", "cpi_yoy", "hpi_yoy"]])
    macro.write_parquet(MACRO_PATH)
    print(f"\nSaved macro data ({macro.height} rows) → {MACRO_PATH}")
    return macro

macro = fetch_macro(force_rebuild=False)
print(macro.head(6))
print(macro.tail(3))

---
## Part A — Exploratory Survival Analysis

### A.1  Overall Kaplan-Meier survival curve and implied hazard rates

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from lifelines import KaplanMeierFitter, NelsonAalenFitter

plt.rcParams.update({"figure.dpi": 130, "font.size": 10})

# Convert to pandas for lifelines (sample down if needed for speed)
MAX_KM_ROWS = 3_000_000   # lifelines handles this comfortably
sv = survival.to_pandas()
if len(sv) > MAX_KM_ROWS:
    sv = sv.sample(MAX_KM_ROWS, random_state=42)
print(f"KM sample size: {len(sv):,}")

T = sv["duration"]
E = sv["prepaid"]   # event = prepayment

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Kaplan-Meier ──────────────────────────────────────────────────────────────
kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E, label="All 30yr FRM")
ax = axes[0]
kmf.plot_survival_function(ax=ax, ci_show=True)
ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Survival Probability  S(t)")
ax.set_title("Kaplan-Meier — Prepayment Survival")
ax.set_xlim(0, 360)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.grid(alpha=0.3)

# ── Nelson-Aalen hazard ───────────────────────────────────────────────────────
naf = NelsonAalenFitter()
naf.fit(T, event_observed=E)

# smooth hazard via rolling mean of cumulative hazard increments
cumhaz = naf.cumulative_hazard_["NA_estimate"]
hazard = cumhaz.diff().fillna(cumhaz.iloc[0])
hazard_smooth = hazard.rolling(6, center=True, min_periods=1).mean()

ax2 = axes[1]
ax2.plot(hazard_smooth.index, hazard_smooth.values * 1000, color="tab:orange", lw=1.5)
ax2.set_xlabel("Loan Age (months)")
ax2.set_ylabel("Hazard Rate (×10⁻³ per month)")
ax2.set_title("Implied Prepayment Hazard Rate (6-month smoothed)")
ax2.set_xlim(0, 360)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "A1_km_hazard.png", bbox_inches="tight")
plt.show()
print(f"Median prepayment time: {kmf.median_survival_time_:.0f} months")

### A.2  Stratified survival curves — LTV, FICO, Vintage

In [ ]:
from lifelines.plotting import add_at_risk_counts

def stratified_km(df, strat_col, bins, labels, title, fname):
    df = df.copy()
    df["_strat"] = pd.cut(df[strat_col], bins=bins, labels=labels, right=False)
    df = df.dropna(subset=["_strat"])

    fig, ax = plt.subplots(figsize=(9, 5))
    fitters = []
    for grp in labels:
        mask = df["_strat"] == grp
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(df.loc[mask, "duration"], event_observed=df.loc[mask, "prepaid"], label=str(grp))
        kmf_g.plot_survival_function(ax=ax, ci_show=False)
        fitters.append(kmf_g)

    ax.set_xlabel("Loan Age (months)")
    ax.set_ylabel("Survival Probability")
    ax.set_title(title)
    ax.set_xlim(0, 360)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(title=strat_col, loc="lower left")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, bbox_inches="tight")
    plt.show()

# ── LTV strata ────────────────────────────────────────────────────────────────
stratified_km(
    sv, "OriginalLoantoValueLTV",
    bins=[0, 60, 80, 95, 105],
    labels=["LTV<60", "60-80", "80-95", "95+"],
    title="Prepayment Survival by LTV Bucket",
    fname="A2_km_ltv.png",
)

# ── FICO strata ───────────────────────────────────────────────────────────────
stratified_km(
    sv, "CreditScore",
    bins=[300, 620, 680, 740, 851],
    labels=["<620", "620-680", "680-740", "740+"],
    title="Prepayment Survival by FICO Bucket",
    fname="A2_km_fico.png",
)

# ── Vintage strata (selected years) ──────────────────────────────────────────
vintage_years = [2006, 2008, 2010, 2012, 2015, 2018, 2020]
fig, ax = plt.subplots(figsize=(9, 5))
for yr in vintage_years:
    mask = sv["VintageYear"] == yr
    if mask.sum() < 500:
        continue
    kmf_v = KaplanMeierFitter()
    kmf_v.fit(sv.loc[mask, "duration"], event_observed=sv.loc[mask, "prepaid"], label=str(yr))
    kmf_v.plot_survival_function(ax=ax, ci_show=False)

ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Survival Probability")
ax.set_title("Prepayment Survival by Origination Vintage")
ax.set_xlim(0, 200)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(title="Vintage", loc="lower left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "A2_km_vintage.png", bbox_inches="tight")
plt.show()

---
## Part B — Classical Cox Proportional Hazards Model

### B.1  Static Cox model

In [ ]:
from lifelines import CoxPHFitter

# ── Feature prep for static Cox ───────────────────────────────────────────────
COX_FEATURES = [
    "CreditScore", "OriginalLoantoValueLTV", "OriginalInterestRate",
    "OriginalDebttoIncomeRatio", "OriginalUPB",
    "LoanPurpose", "OccupancyStatus", "VintageYear",
]

cox_df = sv[COX_FEATURES + ["duration", "prepaid"]].dropna(subset=["duration"])

# Encode categoricals as dummies
cox_df = pd.get_dummies(cox_df, columns=["LoanPurpose", "OccupancyStatus"], drop_first=True)

# Standardize continuous features (Cox is not scale-invariant for display)
from sklearn.preprocessing import StandardScaler
cont_cols = ["CreditScore", "OriginalLoantoValueLTV", "OriginalInterestRate",
             "OriginalDebttoIncomeRatio", "OriginalUPB", "VintageYear"]
scaler = StandardScaler()
cox_df[cont_cols] = scaler.fit_transform(cox_df[cont_cols].fillna(cox_df[cont_cols].median()))

# Subsample for speed if dataset is very large
MAX_COX = 500_000
if len(cox_df) > MAX_COX:
    cox_df = cox_df.sample(MAX_COX, random_state=42)
print(f"Cox training sample: {len(cox_df):,} loans")

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df, duration_col="duration", event_col="prepaid")
cph.print_summary(decimals=4)

In [ ]:
# ── Plot hazard ratios ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_title("Cox PH — Log Hazard Ratios (95% CI)")
ax.set_xlabel("log(Hazard Ratio)  [standardized covariates]")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "B1_cox_hr.png", bbox_inches="tight")
plt.show()

### B.2  Test proportional hazards assumption (Schoenfeld residuals)

In [ ]:
from lifelines.statistics import proportional_hazard_test

SCHOENFELD_PATH = OUT_DIR / "schoenfeld_residuals.parquet"

ph_test = proportional_hazard_test(cph, cox_df, time_transform="rank")
print(ph_test.summary)

if SCHOENFELD_PATH.exists():
    print(f"Loading cached Schoenfeld residuals from {SCHOENFELD_PATH}")
    schoenfeld = pd.read_parquet(SCHOENFELD_PATH)
else:
    print("Computing Schoenfeld residuals (this takes a few minutes) …")
    schoenfeld = cph.compute_residuals(cox_df, kind="scaled_schoenfeld")
    schoenfeld.to_parquet(SCHOENFELD_PATH)
    print(f"Saved → {SCHOENFELD_PATH}")

check_cols = ["CreditScore", "OriginalLoantoValueLTV", "OriginalInterestRate",
              "OriginalDebttoIncomeRatio", "OriginalUPB", "VintageYear"]
check_cols = [c for c in check_cols if c in schoenfeld.columns]

# Subsample residuals only for scatter plot readability
plot_idx = schoenfeld.sample(min(20_000, len(schoenfeld)), random_state=42).index

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.flat, check_cols):
    resids = schoenfeld.loc[plot_idx, col]
    ax.scatter(cox_df.loc[plot_idx, "duration"], resids, s=0.3, alpha=0.3)
    ax.axhline(0, color="red", lw=1)
    ax.set_title(col)
    ax.set_xlabel("Loan Age")
    ax.set_ylabel("Schoenfeld Residual")

plt.suptitle("Scaled Schoenfeld Residuals — PH Assumption Check", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "B2_schoenfeld.png", bbox_inches="tight")
plt.show()

### B.3  Time-varying Cox with macroeconomic covariates

We merge FRED macro data into a monthly panel so each loan-month carries the prevailing mortgage rate, unemployment rate, CPI inflation, and HPI growth at that time.

In [ ]:
from lifelines import CoxTimeVaryingFitter

def build_tv_cox_panel(survival_df: pd.DataFrame, macro_df: pl.DataFrame,
                        n_loans: int = 100_000) -> pd.DataFrame:
    """
    Expand a sample of loans into (start, stop] interval format for
    time-varying Cox, merging macro covariates at each observed month.
    """
    macro_pd = macro_df.to_pandas().set_index("yyyymm")
    sample = survival_df.sample(min(n_loans, len(survival_df)), random_state=42)

    rows = []
    for _, row in sample.iterrows():
        dur   = int(row["duration"])
        event = int(row["prepaid"])
        vpd   = int(row["FirstPaymentDate"])   # YYYYMM of first payment

        for t in range(dur):
            yyyymm = vpd + t // 12 * 100 + (t % 12)
            # handle month overflow
            yr  = yyyymm // 100
            mo  = yyyymm % 100
            if mo > 12:
                yr += mo // 12
                mo  = mo % 12
                if mo == 0:
                    yr -= 1
                    mo  = 12
            yyyymm = yr * 100 + mo

            macro_row = macro_pd.loc[yyyymm] if yyyymm in macro_pd.index else None

            rows.append({
                "id":            row["LoanSequenceNumber"],
                "start":         t,
                "stop":          t + 1,
                "event":         1 if (t == dur - 1 and event == 1) else 0,
                "CreditScore":   row["CreditScore"],
                "LTV":           row["OriginalLoantoValueLTV"],
                "OrigRate":      row["OriginalInterestRate"],
                "LoanPurpose":   row["LoanPurpose"],
                "mortgage_rate": macro_row["mortgage_rate"] if macro_row is not None else np.nan,
                "unemployment":  macro_row["unemployment"]  if macro_row is not None else np.nan,
                "cpi_yoy":       macro_row["cpi_yoy"]       if macro_row is not None else np.nan,
                "hpi_yoy":       macro_row["hpi_yoy"]       if macro_row is not None else np.nan,
            })

    panel = pd.DataFrame(rows)
    # rate incentive = orig_rate - current_mortgage_rate (key prepayment driver)
    panel["rate_incentive"] = panel["OrigRate"] - panel["mortgage_rate"]
    panel = pd.get_dummies(panel, columns=["LoanPurpose"], drop_first=True)
    return panel.dropna()

print("Building time-varying Cox panel (100K loans) …")
tv_panel = build_tv_cox_panel(sv, macro)
print(f"  Panel rows: {len(tv_panel):,}")

tv_feat_cols = [c for c in tv_panel.columns
                if c not in ("id", "start", "stop", "event")]

ctv = CoxTimeVaryingFitter(penalizer=0.1)
ctv.fit(tv_panel, id_col="id", start_col="start", stop_col="stop",
        event_col="event", formula=" + ".join(tv_feat_cols))
ctv.print_summary(decimals=4)

---
## Step 3 — Build ML Panel Dataset

Expand a 2M-loan stratified sample into a monthly panel (one row per loan-month), then join macro covariates. This is the training data for Parts C and D.

In [ ]:
def build_ml_panel(survival_pl: pl.DataFrame, macro_pl: pl.DataFrame,
                   n_sample: int = 2_000_000,
                   force_rebuild: bool = False) -> None:
    """
    Builds the ML discrete-time panel and writes it to PANEL_PATH.
    One row per (loan, month): outcome = prepaid_this_month (binary).
    """
    if PANEL_PATH.exists() and not force_rebuild:
        print(f"Panel already exists at {PANEL_PATH}. Set force_rebuild=True to regenerate.")
        return

    # Stratified sample: proportional to vintage year counts
    sample = (
        survival_pl
        .with_columns(pl.col("VintageYear").cast(pl.Int32))
        .group_by("VintageYear")
        .map_groups(lambda g: g.sample(
            n=min(len(g), max(1, int(n_sample * len(g) / survival_pl.height))),
            seed=42
        ))
    )
    print(f"Sampled {sample.height:,} loans for ML panel")

    # Static feature columns to carry through
    STATIC_COLS = [
        "LoanSequenceNumber", "CreditScore", "OriginalLoantoValueLTV",
        "OriginalInterestRate", "OriginalDebttoIncomeRatio", "OriginalUPB",
        "LoanPurpose", "OccupancyStatus", "VintageYear",
        "FirstPaymentDate", "duration", "prepaid",
    ]
    sample_pd = sample.select(STATIC_COLS).to_pandas()
    macro_pd  = macro_pl.to_pandas().set_index("yyyymm")

    chunks = []
    CHUNK = 50_000

    for start_idx in range(0, len(sample_pd), CHUNK):
        batch = sample_pd.iloc[start_idx : start_idx + CHUNK]
        batch_rows = []
        for _, row in batch.iterrows():
            dur   = int(row["duration"])
            event = int(row["prepaid"])
            fpd   = int(row["FirstPaymentDate"])

            for t in range(dur):
                yr = fpd // 100 + (fpd % 100 + t - 1) // 12
                mo = (fpd % 100 + t - 1) % 12 + 1
                yyyymm = yr * 100 + mo
                m = macro_pd.loc[yyyymm] if yyyymm in macro_pd.index else None

                batch_rows.append({
                    "loan_id":        row["LoanSequenceNumber"],
                    "loan_age":       t,
                    "prepaid_month":  1 if (t == dur - 1 and event == 1) else 0,
                    "FICO":           row["CreditScore"],
                    "LTV":            row["OriginalLoantoValueLTV"],
                    "orig_rate":      row["OriginalInterestRate"],
                    "DTI":            row["OriginalDebttoIncomeRatio"],
                    "UPB":            row["OriginalUPB"],
                    "loan_purpose":   row["LoanPurpose"],
                    "occupancy":      row["OccupancyStatus"],
                    "vintage_year":   row["VintageYear"],
                    "mortgage_rate":  m["mortgage_rate"]  if m is not None else np.nan,
                    "unemployment":   m["unemployment"]   if m is not None else np.nan,
                    "cpi_yoy":        m["cpi_yoy"]        if m is not None else np.nan,
                    "hpi_yoy":        m["hpi_yoy"]        if m is not None else np.nan,
                    "rate_incentive": row["OriginalInterestRate"] - (m["mortgage_rate"] if m is not None else np.nan),
                })

        chunk_df = pl.from_pandas(pd.DataFrame(batch_rows))
        chunks.append(chunk_df)
        print(f"  Processed {min(start_idx + CHUNK, len(sample_pd)):,} / {len(sample_pd):,} loans", end="\r")

    print()
    panel = pl.concat(chunks)
    panel.write_parquet(PANEL_PATH)
    print(f"  Saved ML panel: {panel.height:,} rows → {PANEL_PATH}")

build_ml_panel(survival, macro, n_sample=2_000_000, force_rebuild=False)

---
## Part C — Machine Learning Models

Train/val/test split by origination vintage:
- **Train**: vintage 1999–2016  
- **Val**: vintage 2017–2019  
- **Test**: vintage 2020–2022  

Models: XGBoost, LightGBM, Elastic Net logistic regression.  
Evaluation: C-index, time-dependent AUC, Brier score.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from lifelines.utils import concordance_index

# ── Load panel with polars, split by vintage, then subsample ──────────────────
# Stay in polars until after sampling to avoid OOM crash in pandas conversion.
CAT_COLS = ["loan_purpose", "occupancy"]
MAX_TRAIN = 5_000_000
MAX_VALTE = 1_000_000

panel_pl = pl.read_parquet(PANEL_PATH)
panel_pl = panel_pl.with_columns([
    pl.col(c).cast(pl.Utf8) for c in CAT_COLS if c in panel_pl.columns
])

def sample_split(df: pl.DataFrame, max_rows: int) -> pl.DataFrame:
    """Keep all events; downsample non-events to stay within max_rows."""
    events   = df.filter(pl.col("prepaid_month") == 1)
    nonevents = df.filter(pl.col("prepaid_month") == 0)
    n_keep = max(0, max_rows - events.height)
    if nonevents.height > n_keep:
        nonevents = nonevents.sample(n_keep, seed=42)
    return pl.concat([events, nonevents]).sample(fraction=1.0, seed=42)  # shuffle

train_pl = sample_split(panel_pl.filter(pl.col("vintage_year") <= 2016), MAX_TRAIN)
val_pl   = sample_split(panel_pl.filter((pl.col("vintage_year") >= 2017) & (pl.col("vintage_year") <= 2019)), MAX_VALTE)
test_pl  = sample_split(panel_pl.filter(pl.col("vintage_year") >= 2020), MAX_VALTE)
del panel_pl  # free memory

# Convert to pandas
train = train_pl.to_pandas(); del train_pl
val   = val_pl.to_pandas();   del val_pl
test  = test_pl.to_pandas();  del test_pl

train = pd.get_dummies(train, columns=CAT_COLS, drop_first=True)
val   = pd.get_dummies(val,   columns=CAT_COLS, drop_first=True)
test  = pd.get_dummies(test,  columns=CAT_COLS, drop_first=True)

# Align columns (dummies may differ across splits)
train, val  = train.align(val,  join="left", axis=1, fill_value=0)
train, test = train.align(test, join="left", axis=1, fill_value=0)

FEATURES = [
    "loan_age", "FICO", "LTV", "orig_rate", "DTI", "UPB",
    "vintage_year", "mortgage_rate", "unemployment", "cpi_yoy", "hpi_yoy",
    "rate_incentive",
] + [c for c in train.columns if c.startswith("loan_purpose_") or c.startswith("occupancy_")]
FEATURES = [f for f in FEATURES if f in train.columns]

TARGET = "prepaid_month"

X_tr, y_tr = train[FEATURES].fillna(0), train[TARGET]
X_va, y_va = val[FEATURES].fillna(0),   val[TARGET]
X_te, y_te = test[FEATURES].fillna(0),  test[TARGET]

print(f"Train: {len(X_tr):>9,} rows  ({y_tr.mean():.3%} event rate)")
print(f"Val  : {len(X_va):>9,} rows  ({y_va.mean():.3%} event rate)")
print(f"Test : {len(X_te):>9,} rows  ({y_te.mean():.3%} event rate)")

In [ ]:
def eval_model(name, model, X_te, y_te, duration_te):
    proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_te)
    auc   = roc_auc_score(y_te, proba)
    brier = brier_score_loss(y_te, proba)
    # C-index: higher predicted prepayment prob → shorter survival time
    ci    = concordance_index(duration_te, -proba, y_te)
    print(f"  {name:<20}  AUC={auc:.4f}  Brier={brier:.5f}  C-index={ci:.4f}")
    return {"model": name, "AUC": auc, "Brier": brier, "C-index": ci, "proba": proba}

results = []
dur_te = test["loan_age"]   # proxy for duration at observation

# ── XGBoost ───────────────────────────────────────────────────────────────────
print("Training XGBoost …")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
    tree_method="hist", device="cpu",
    eval_metric="auc", early_stopping_rounds=20,
    random_state=42, n_jobs=-1,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=50)
results.append(eval_model("XGBoost", xgb_model, X_te, y_te, dur_te))

# ── LightGBM ──────────────────────────────────────────────────────────────────
print("Training LightGBM …")
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(50)])
results.append(eval_model("LightGBM", lgb_model, X_te, y_te, dur_te))

# ── Random Forest ─────────────────────────────────────────────────────────────
print("Training Random Forest …")
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight="balanced", random_state=42, n_jobs=-1,
)
rf_model.fit(X_tr, y_tr)
results.append(eval_model("Random Forest", rf_model, X_te, y_te, dur_te))

# ── Elastic Net ───────────────────────────────────────────────────────────────
print("Training Elastic Net …")
scaler_ml = StandardScaler()
X_tr_s = scaler_ml.fit_transform(X_tr)
X_te_s  = scaler_ml.transform(X_te)
enet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.1,
                          max_iter=500, random_state=42, n_jobs=-1)
enet.fit(X_tr_s, y_tr)
results.append(eval_model("Elastic Net", enet, X_te_s, y_te, dur_te))

# ── Cox C-index benchmark ─────────────────────────────────────────────────────
cox_feat_cols = [c for c in cox_df.columns if c not in ("duration", "prepaid")]
# Use static Cox score on test loans that overlap
print("\nSummary:")
print(f"  {'Model':<20}  {'AUC':>6}  {'Brier':>7}  {'C-index':>8}")
for r in results:
    print(f"  {r['model']:<20}  {r['AUC']:.4f}  {r['Brier']:.5f}  {r['C-index']:.4f}")

In [ ]:
# ── Feature importance ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
imp_xgb.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("XGBoost Feature Importance (gain)")
axes[0].set_xlabel("Importance")

imp_lgb = pd.Series(lgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
imp_lgb.plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("LightGBM Feature Importance (split)")
axes[1].set_xlabel("Importance")

imp_rf = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(15)
imp_rf.plot(kind="barh", ax=axes[2], color="forestgreen")
axes[2].set_title("Random Forest Feature Importance (Gini)")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig(OUT_DIR / "C_feature_importance.png", bbox_inches="tight")
plt.show()


---
## Part D — Deep Cox Model

Replace the linear predictor in Cox with a neural network:

$$\lambda(t \mid X) = \lambda_0(t) \exp\bigl(f_\theta(X)\bigr)$$

where $f_\theta$ is an MLP trained via the Breslow partial likelihood.  
Uses PyTorch with the M1 MPS backend.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("cpu")
)
print(f"Using device: {DEVICE}")

# ── Dataset: one row per loan (static features only) ─────────────────────────
# We use the survival-level dataset, not the panel, for the partial likelihood.
DC_N = 500_000
dc_static = ["CreditScore", "OriginalLoantoValueLTV", "OriginalInterestRate",
             "OriginalDebttoIncomeRatio", "OriginalUPB", "VintageYear"]

dc_df = sv[dc_static + ["duration", "prepaid"]].dropna().sample(DC_N, random_state=42)

# Join macro at origination date (rate environment when loan was made)
dc_df = dc_df.merge(
    macro.to_pandas().rename(columns={"yyyymm": "FirstPaymentDate"}),
    left_on="VintageYear",  # approximate: use vintage year avg
    right_on=macro.to_pandas()["yyyymm"].apply(lambda x: x // 100),
    how="left",
).dropna()

feat_cols_dc = dc_static + ["mortgage_rate", "unemployment", "cpi_yoy", "hpi_yoy"]
scaler_dc = StandardScaler()
X_dc = scaler_dc.fit_transform(dc_df[feat_cols_dc].fillna(dc_df[feat_cols_dc].median()))
T_dc = dc_df["duration"].values.astype(np.float32)
E_dc = dc_df["prepaid"].values.astype(np.float32)

# Sort by descending duration (required for Breslow partial likelihood)
sort_idx = np.argsort(-T_dc)
X_dc, T_dc, E_dc = X_dc[sort_idx], T_dc[sort_idx], E_dc[sort_idx]

# Train/test split (80/20 by index, not vintage, to keep it simple for Deep Cox)
n_train = int(0.8 * len(X_dc))
X_tr_dc, X_te_dc = X_dc[:n_train], X_dc[n_train:]
T_tr_dc, T_te_dc = T_dc[:n_train], T_dc[n_train:]
E_tr_dc, E_te_dc = E_dc[:n_train], E_dc[n_train:]

X_tr_t = torch.tensor(X_tr_dc, dtype=torch.float32)
X_te_t = torch.tensor(X_te_dc, dtype=torch.float32)
T_tr_t = torch.tensor(T_tr_dc, dtype=torch.float32)
E_tr_t = torch.tensor(E_tr_dc, dtype=torch.float32)

print(f"Deep Cox train: {len(X_tr_dc):,}  test: {len(X_te_dc):,}")

In [ ]:
class DeepCox(nn.Module):
    def __init__(self, in_features: int, hidden: list[int] = [128, 64, 32], dropout: float = 0.2):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))   # scalar log-hazard ratio f_θ(X)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def breslow_partial_likelihood(log_hz: torch.Tensor, event: torch.Tensor) -> torch.Tensor:
    """
    Negative Breslow partial log-likelihood.
    Input must be sorted by DESCENDING survival time.
    """
    log_cumsum = torch.logcumsumexp(log_hz, dim=0)
    loss = -torch.mean((log_hz - log_cumsum) * event)
    return loss


# ── Training ──────────────────────────────────────────────────────────────────
in_dim = X_tr_t.shape[1]
model_dc = DeepCox(in_dim, hidden=[256, 128, 64], dropout=0.3).to(DEVICE)
optimizer = torch.optim.Adam(model_dc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

BATCH   = 4096
EPOCHS  = 40
dataset = TensorDataset(X_tr_t, E_tr_t)
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True)

train_losses = []
for epoch in range(1, EPOCHS + 1):
    model_dc.train()
    epoch_loss = 0.0
    for X_batch, E_batch in loader:
        X_batch, E_batch = X_batch.to(DEVICE), E_batch.to(DEVICE)
        optimizer.zero_grad()
        log_hz = model_dc(X_batch)
        loss   = breslow_partial_likelihood(log_hz, E_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg = epoch_loss / len(loader)
    train_losses.append(avg)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS}  loss={avg:.5f}")

plt.figure(figsize=(7, 3))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Negative Partial Log-Likelihood")
plt.title("Deep Cox Training Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "D_training_loss.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Evaluate Deep Cox ─────────────────────────────────────────────────────────
model_dc.eval()
with torch.no_grad():
    log_hz_te = model_dc(X_te_t.to(DEVICE)).cpu().numpy()

ci_dc = concordance_index(T_te_dc, -log_hz_te, E_te_dc)
print(f"Deep Cox C-index (test): {ci_dc:.4f}")

# ── Model comparison table ────────────────────────────────────────────────────
cox_ci = cph.concordance_index_
print(f"\n{'Model':<20}  {'C-index':>8}")
print(f"{'Cox PH (static)':<20}  {cox_ci:.4f}")
for r in results:
    print(f"{r['model']:<20}  {r['C-index']:.4f}")
print(f"{'Deep Cox':<20}  {ci_dc:.4f}")

### D.v  Nonlinear Interactions — Gradient Sensitivity Analysis

Compute the average absolute gradient $|\partial f_\theta / \partial x_j|$ over the test set.
This gives a Deep Cox analogue of Cox hazard ratios, capturing how much each input
drives the predicted log-hazard — including nonlinear and interaction effects.


In [ ]:
# ── D.v Gradient sensitivity: |∂f_θ/∂x_j| averaged over test set ────────────
model_dc.eval()
X_te_grad = torch.tensor(X_te_dc, dtype=torch.float32, requires_grad=True).to(DEVICE)
log_hz = model_dc(X_te_grad)
log_hz.sum().backward()  # accumulate grads w.r.t. all inputs

grad_sensitivity = X_te_grad.grad.abs().cpu().numpy().mean(axis=0)
feat_sens = pd.Series(grad_sensitivity, index=feat_cols_dc).sort_values(ascending=False)

print("Gradient sensitivity (most → least influential):")
for feat, val in feat_sens.items():
    print(f"  {feat:<35}: {val:.5f}")

fig, ax = plt.subplots(figsize=(8, 4))
feat_sens.sort_values().plot(kind="barh", ax=ax, color="mediumpurple")
ax.set_title("Deep Cox — Gradient Sensitivity $|\partial f_\theta/\partial x_j|$")
ax.set_xlabel("Mean |Gradient| over test set")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(OUT_DIR / "D_gradient_sensitivity.png", bbox_inches="tight")
plt.show()


---
## Part E — Extensions

### E.1  Competing Risks: Prepayment vs. Default

Use the Aalen-Johansen estimator for cumulative incidence functions (CIF).  
Event codes: prepayment = 1, default = {2, 3, 9}, censored = everything else.

In [ ]:
from lifelines import AalenJohansenFitter

# event_type: 1 = prepaid, 2 = defaulted, 0 = censored
sv_cr = sv.copy()
sv_cr["event_type"] = 0
sv_cr.loc[sv_cr["prepaid"] == 1,   "event_type"] = 1
sv_cr.loc[sv_cr["defaulted"] == 1, "event_type"] = 2

T_cr = sv_cr["duration"]
E_cr = sv_cr["event_type"]

fig, ax = plt.subplots(figsize=(9, 5))

# ── Prepayment CIF ────────────────────────────────────────────────────────────
ajf_prepay = AalenJohansenFitter()
ajf_prepay.fit(T_cr, E_cr, event_of_interest=1, label="Prepayment")
ajf_prepay.plot_cumulative_density(ax=ax)

# ── Default CIF ───────────────────────────────────────────────────────────────
ajf_default = AalenJohansenFitter()
ajf_default.fit(T_cr, E_cr, event_of_interest=2, label="Default")
ajf_default.plot_cumulative_density(ax=ax)

ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Cumulative Incidence")
ax.set_title("Competing Risks: Cumulative Incidence Functions")
ax.set_xlim(0, 360)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "E1_competing_risks_cif.png", bbox_inches="tight")
plt.show()

col_p = ajf_prepay.cumulative_density_.columns[0]
col_d = ajf_default.cumulative_density_.columns[0]
print(f"10-yr prepayment CIF : {ajf_prepay.cumulative_density_[col_p].loc[120]:.3f}")
print(f"10-yr default    CIF : {ajf_default.cumulative_density_[col_d].loc[120]:.4f}")

### E.2  Scenario Analysis — Interest Rate Shocks

Shock the mortgage rate covariate and observe the change in predicted prepayment hazard from the Deep Cox and XGBoost models.

In [ ]:
SHOCKS_BP = [-200, -100, 0, +100, +200]   # basis points

# ── Reference feature set: 2000 test loans at baseline ───────────────────────
test_dc = dc_df.iloc[n_train:n_train + 2000].copy()
base_features = scaler_dc.transform(test_dc[feat_cols_dc].fillna(test_dc[feat_cols_dc].median()))
mortgage_rate_idx = feat_cols_dc.index("mortgage_rate")

results_scenario = {}
model_dc.eval()

for shock_bp in SHOCKS_BP:
    shocked = base_features.copy()
    shocked[:, mortgage_rate_idx] += shock_bp / 100.0   # add shock in raw units

    # Deep Cox risk score
    with torch.no_grad():
        log_hz_shock = model_dc(torch.tensor(shocked, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    results_scenario[shock_bp] = {"DeepCox_log_hz": log_hz_shock.mean()}

    # XGBoost: use 2000 val rows with non-NaN mortgage_rate; apply shock + recalc rate_incentive
    panel_sample = val[val["mortgage_rate"].notna()].head(2000).copy()
    if len(panel_sample) > 0 and "mortgage_rate" in panel_sample.columns:
        panel_sample["mortgage_rate"] = panel_sample["mortgage_rate"] + shock_bp / 100.0
        panel_sample["rate_incentive"] = panel_sample["orig_rate"] - panel_sample["mortgage_rate"]
        feat_data = panel_sample[FEATURES].copy()
        for col in FEATURES:
            if col not in feat_data.columns:
                feat_data[col] = 0.0
        xgb_proba = xgb_model.predict_proba(feat_data.fillna(feat_data.median()))[:, 1].mean()
    else:
        xgb_proba = np.nan
    results_scenario[shock_bp]["XGB_avg_proba"] = xgb_proba

# ── Plot ──────────────────────────────────────────────────────────────────────
shocks  = SHOCKS_BP
dc_scores = [results_scenario[s]["DeepCox_log_hz"] for s in shocks]
xgb_proba = [results_scenario[s]["XGB_avg_proba"] for s in shocks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(shocks, dc_scores, "o-", color="tab:blue")
axes[0].axvline(0, ls="--", color="gray", lw=0.8)
axes[0].set_xlabel("Rate Shock (bp)")
axes[0].set_ylabel("Mean Log-Hazard Ratio")
axes[0].set_title("Deep Cox: Hazard Response to Rate Shock")
axes[0].grid(alpha=0.3)

axes[1].plot(shocks, xgb_proba, "s-", color="tab:orange")
axes[1].axvline(0, ls="--", color="gray", lw=0.8)
axes[1].set_xlabel("Rate Shock (bp)")
axes[1].set_ylabel("Avg Monthly Prepayment Probability")
axes[1].set_title("XGBoost: Prepayment Response to Rate Shock")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=3))
axes[1].grid(alpha=0.3)

plt.suptitle("Scenario Analysis: Prepayment Sensitivity to Interest Rate Shocks", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "E2_scenario_analysis.png", bbox_inches="tight")
plt.show()

print("\nScenario Summary:")
print(f"{'Shock (bp)':<12} {'DeepCox logHR':>15} {'XGB Prob':>12}")
for s in shocks:
    r = results_scenario[s]
    print(f"{s:>+12}  {r['DeepCox_log_hz']:>15.4f}  {r['XGB_avg_proba']:>12.5f}")